# FAIR 데이터 원칙 실습

**FAIR Data · 찾기·접근·상호운용·재사용**

데이터를 찾을 수 있고, 접근할 수 있고, 함께 쓸 수 있고, 재사용할 수 있게 만드는 관리 원칙.

소재 분야에서 이해하기: 측정 조건과 단위를 표준 항목으로 기록해 공개한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [FAIR 데이터 원칙](https://www.go-fair.org/fair-principles/)

## 1. 메타데이터 없는 기록의 문제

같은 측정값이라도 단위와 조건이 없으면 재사용할 수 없습니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

poor_record = {'sample': 'A-12', 'value': 431}
rich_record = {
    'sample_id': 'A-12',
    'identifier': 'https://doi.org/10.5281/zenodo.0000000',
    'property': 'vickers_hardness',
    'value': 431.0, 'unit': 'HV',
    'uncertainty': 12.0, 'uncertainty_type': 'standard_deviation',
    'method': 'ISO 6507-1, 10 kgf, 15 s',
    'composition': {'Fe': 0.72, 'Cr': 0.18, 'Ni': 0.10},
    'process': {'sintering_temperature_C': 780, 'hold_time_h': 4.0},
    'operator_role': 'lab_technician',
    'measured_on': '2026-09-01',
    'license': 'CC-BY-4.0',
}
print('빈약한 기록:', poor_record)
print('\n무엇이 431인지, 어떤 단위인지, 어떤 조건인지 알 수 없어 다른 연구와 합칠 수 없습니다.')

## 2. 필수 항목 검증기

FAIR의 F·A·I·R을 최소한으로 점검하는 검사기를 만들어 봅니다.

In [ ]:
REQUIRED = {
    'Findable': ['sample_id', 'identifier'],
    'Accessible': ['license'],
    'Interoperable': ['property', 'unit', 'method'],
    'Reusable': ['value', 'uncertainty', 'composition', 'process', 'measured_on'],
}

def audit(record):
    report = {}
    for principle, fields in REQUIRED.items():
        missing = [field for field in fields if field not in record or record[field] in (None, '')]
        report[principle] = missing
    return report

for name, record in [('빈약한 기록', poor_record), ('충실한 기록', rich_record)]:
    print(name)
    for principle, missing in audit(record).items():
        print('   %-14s %s' % (principle, '통과' if not missing else '누락 ' + ', '.join(missing)))

## 3. 단위 변환과 상호운용성

In [ ]:
import pandas as pd

records = [
    {'sample_id': 'A-1', 'property': 'hardness', 'value': 431.0, 'unit': 'HV'},
    {'sample_id': 'A-2', 'property': 'hardness', 'value': 4.23, 'unit': 'GPa'},
    {'sample_id': 'A-3', 'property': 'hardness', 'value': 455.0, 'unit': 'HV'},
]
CONVERSION = {'HV': 1.0, 'GPa': 1000.0 / 9.807}      # 1 GPa ~= 102 HV (근사)

table = pd.DataFrame(records)
table['value_HV'] = [row.value * CONVERSION[row.unit] for row in table.itertuples()]
print(table)
print('\n단위를 기록하지 않았다면 4.23 을 431 과 같은 축에 올릴 방법이 없습니다.')
print('공개 시에는 식별자와 라이선스를 함께 붙여 재사용 조건을 분명히 합니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#fair-data)을 여세요.